In [1]:
USE_KAGGLE_SANDBOX = True
KAGGLE_SANDBOX_KWARGS = {
    'dataset_data_sources': ['greynearing/camels-us-531-era'],
    'enable_gpu': True,
    'enable_internet': False,
    'machine_shape': 'NvidiaTeslaT4Highmem',
}
C_PUCT = 2.5

# CAMELS-US ERA — Gauged Physics-Informed

## Objective
Maximize the **Median Nash-Sutcliffe Efficiency (NSE)** across all **531 CAMELS-US basins** on the out-of-sample evaluation period (`1999-10-01` to `2003-09-30`, 4 water years, `1,461` days).

## Data & Split Protocol
- **Spatial Regime (Gauged)**: All 531 basins have training streamflow observations `QOBS_TRAIN` (`mm/day`) available during the weights training period (`2003-10-01` to `2008-09-30`, 5 water years, `1,827` days).
- **Strict Newman Temporal Containment (`1998-10-01` to `2008-09-30`, `3,653` days)**:
  - `1998-10-01` to `1999-09-30` (`365` days): 1-year meteorological spin-up prior to evaluation (`QOBS_TRAIN = NaN`).
  - `1999-10-01` to `2003-09-30` (`1,461` days, `ERA_EVAL_MASK`): Out-of-sample ERA hill-climb evaluation period (`QOBS_TRAIN = NaN`). The final 365 days (`2002-10-01` to `2003-09-30`) also serve as the 1-year meteorological spin-up prior to `2003-10-01`.
  - `2003-10-01` to `2008-09-30` (`1,827` days, `ERA_TRAIN_MASK`): ERA model weights training period (`QOBS_TRAIN` observed).
- **Inputs Available**:
  - `DYNAMIC_INPUTS`: shape `(531, 3653, 15)` (`float32`) — 5 daily meteorological variables (`prcp`, `srad`, `tmax`, `tmin`, `vp`) across 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
  - `AUX_FORCINGS`: shape `(531, 3653, 6)` (`float32`) — daily `dayl` and `swe` across `Daymet`, `Maurer`, `NLDAS`.
  - `STATIC_ATTRIBUTES`: shape `(531, 27)` (`float32`) — 27 catchment climate, topography, soil, geology, and vegetation attributes (plus `BASIN_AREAS`, `LAT`, `LON`, `HUC_02`).
- **Required Output**:
  - The mutable cell must produce `Q_PRED` as a `float32` NumPy array of shape `(531, 3653)` containing daily streamflow predictions (`mm/day`).

## Hydrological Physics & Architectural Guidance
You are encouraged to integrate physical hydrological principles and domain knowledge into the model architecture, feature engineering, and training objective:
1. **Conservation Principles & Mass-Conserving Architectures**: Consider mass-conserving recurrent structures (e.g., MC-LSTM) or explicit conceptual-neural storage cells that partition precipitation and snowmelt into tracked storage states and outgoing fluxes (evapotranspiration, quickflow, interflow, slow groundwater baseflow).
2. **Snow Accumulation, Melt & Soil Moisture Accounting**: Incorporate explicit degree-day / radiation-temperature snowpack accounting (tracking snow water equivalent SWE and liquid water delivery) and multi-rate exponential Antecedent Precipitation / Soil Moisture Indices (API).
3. **Catchment Conditioning (FiLM / Hypernetworks)**: Use static catchment attributes (aridity, snow fraction, soil conductivity, geology, slope, vegetation) to modulate hidden dynamics via Feature-wise Linear Modulation (FiLM) or regularized hypernetworks.
4. **Multi-Forcing Fusion & Input Uncertainty**: Leverage the 3 meteorological forcing products (`Daymet`, `Maurer`, `NLDAS`) and auxiliary SWE/daylength inputs by learning product-uncertainty weights and cross-forcing agreement features.

In [2]:
import os
import time
import math
import json
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

_SEARCH_DIRS = [
    Path("/kaggle/input/camels-us-531-era"),
    Path("/kaggle/input/datasets/greynearing/camels-us-531-era"),
    Path(__file__).resolve().parent.parent / "splits" if "__file__" in globals() else Path("../splits"),
    Path("../splits"),
    Path("./splits"),
]

def _find_file(filename: str) -> Path:
    for d in _SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {filename} in any of {_SEARCH_DIRS}")

_train_npz = np.load(_find_file("camels_era_train.npz"), allow_pickle=True)
_basins_json = json.loads(_find_file("basins_531.json").read_text(encoding="utf-8"))

BASINS = _train_npz["basins"].astype(str)
_canonical_basins = [str(b) for b in _basins_json]
assert list(BASINS) == _canonical_basins, "BASINS order mismatch against canonical basins_531.json"

# Strictly enforce containment to the Newman training window (1998-10-01 to 2008-09-30, 3653 days)
_raw_dates = np.array([str(d) for d in _train_npz["dates"]])
_newman_window = (_raw_dates >= "1998-10-01") & (_raw_dates <= "2008-09-30")
DATES = _raw_dates[_newman_window]
assert len(DATES) == 3653 and DATES[0] == "1998-10-01" and DATES[-1] == "2008-09-30"

DYNAMIC_INPUTS = _train_npz["dynamic_inputs"][:, _newman_window, :].astype(np.float32)
DYNAMIC_FEATURE_NAMES = [str(x) for x in _train_npz["dynamic_feature_names"]]
AUX_FORCINGS = _train_npz["aux_forcings"][:, _newman_window, :].astype(np.float32)
AUX_FEATURE_NAMES = [str(x) for x in _train_npz["aux_feature_names"]]
STATIC_ATTRIBUTES = _train_npz["static_attributes"].astype(np.float32)
STATIC_ATTRIBUTE_NAMES = [str(x) for x in _train_npz["static_attribute_names"]]

ERA_TRAIN_MASK = (DATES >= "2003-10-01") & (DATES <= "2008-09-30")
ERA_EVAL_MASK = (DATES >= "1999-10-01") & (DATES <= "2003-09-30")
assert int(ERA_TRAIN_MASK.sum()) == 1827, f"Expected 1827 training days, got {int(ERA_TRAIN_MASK.sum())}"
assert int(ERA_EVAL_MASK.sum()) == 1461, f"Expected 1461 eval days, got {int(ERA_EVAL_MASK.sum())}"

QOBS_TRAIN = np.where(
    ERA_TRAIN_MASK[None, :],
    _train_npz["qobs_train"][:, _newman_window].astype(np.float32),
    np.nan,
).astype(np.float32)

BASIN_AREAS = _train_npz["basin_areas"].astype(np.float32)
LAT = _train_npz["lat"].astype(np.float32)
LON = _train_npz["lon"].astype(np.float32)

SPATIAL_REGIME = "gauged"
print(f"Loaded CAMELS-US 531 dataset ({SPATIAL_REGIME}): {len(BASINS)} basins, {len(DATES)} days ({DATES[0]} to {DATES[-1]}).")
print(f"ERA_TRAIN_MASK (2003-10-01..2008-09-30): {int(ERA_TRAIN_MASK.sum())} days | ERA_EVAL_MASK (1999-10-01..2003-09-30): {int(ERA_EVAL_MASK.sum())} days")
print(f"Dynamic inputs: {DYNAMIC_INPUTS.shape}, Aux: {AUX_FORCINGS.shape}, Static: {STATIC_ATTRIBUTES.shape}")
print(f"Available GPUs: {NUM_GPUS} (Primary device: {DEVICE})")

Loaded CAMELS-US 531 dataset (gauged): 531 basins, 3653 days (1998-10-01 to 2008-09-30).
ERA_TRAIN_MASK (2003-10-01..2008-09-30): 1827 days | ERA_EVAL_MASK (1999-10-01..2003-09-30): 1461 days
Dynamic inputs: (531, 3653, 15), Aux: (531, 3653, 6), Static: (531, 27)
Available GPUs: 2 (Primary device: cuda)


In [3]:
# Begin Stage

In [4]:
INSTRUCTION = [
    "Explore a Mass-Conserving LSTM (MC-LSTM) or water-balance-constrained recurrent cell where input precipitation is explicitly partitioned into conserved storage states and outgoing fluxes (evapotranspiration, quickflow, baseflow), combined in an ensemble with the gated hydrological LSTM.",
    "Design a differentiable conceptual-neural hybrid model (inspired by dPL / delta-HBV / GR4J) where a neural network predicts dynamic or basin-specific physical parameters (snowmelt degree-day factor, soil moisture capacity beta, percolation rate, multi-reservoir routing time constants) integrated with the deep sequence model.",
    "Build a multi-timescale hydrological architecture (inspired by MTS-LSTM) or dilated causal Temporal Convolutional Network (TCN / WaveNet) branch that processes long-term 365-day groundwater/snowpack memory alongside high-frequency daily rainfall-runoff dynamics.",
    "Incorporate an explicit snow accumulation and melt conceptual module (tracking snow water equivalent SWE state via temperature-index / radiation-temperature melt equations) and feed its simulated liquid water input (rain + snowmelt) and snowpack state into the neural network.",
    "Explore a multi-bucket routing head that decomposes streamflow into physically distinct pathways (surface overland flow, interflow, slow groundwater baseflow) with different nonlinear activation dynamics and exponential recession filters.",
    "Implement Feature-wise Linear Modulation (FiLM) or a Hypernetwork where static catchment attributes (climate, geology, soil permeability, topography) dynamically modulate both the input projections and intermediate hidden layer activations.",
    "Engineer physically grounded hydroclimatic features: saturation vapor pressure, relative humidity, net radiation energy balance proxy, aridity sensitivity, multi-window exponential Antecedent Soil Moisture / Precipitation Indices (API with learned or multi-rate decay constants), and freezing-thawing degree days.",
    "Synthesize the best-performing physical-neural hybrid components (explicit snow/soil-moisture accounting, FiLM static modulation, multi-scale recurrent/convolutional pathways, and multi-pathway routing) into a diverse, well-regularized multi-model ensemble."
]
VALUE_REGEX = r'(?m)^Candidate metric to be maximized: (.*)$'
OBJECTIVE = 'maximize'
WORST_VALUE = -2e308

In [5]:
# [rewrite_cell]
# %% [rewrite_cell]
# ==============================================================================
# PHYSICS-INFORMED DIFFERENTIABLE EA-LSTM (d-EA-LSTM)
# ==============================================================================

# Hyperparameters
SEQ_LEN = 365
HIDDEN_SIZE = 256
EMBED_DIM = 64
DROPOUT = 0.4
BATCH_SIZE = 128
LR = 1.5e-3
EPOCHS = 50
NUM_MODELS = 3
SAMPLES_PER_BASIN = 25
WARMUP_LEN = 180

class DifferentiableSnowPack(nn.Module):
    """Differentiable degree-day snow module."""
    def forward(self, p, t, melt_factor, t_crit):
        # p: (B, T), t: (B, T), melt_factor/t_crit: (B, 1)
        b, l = p.shape
        snow_pack = torch.zeros(b, device=p.device)
        liquid_delivery = torch.zeros(b, l, device=p.device)
        
        for i in range(l):
            temp = t[:, i]
            precip = p[:, i]
            
            # Partition rain vs snow
            is_snow = (temp < t_crit.squeeze()).float()
            snowfall = precip * is_snow
            rainfall = precip * (1 - is_snow)
            
            # Potential melt
            melt = torch.clamp(melt_factor.squeeze() * (temp - t_crit.squeeze()), min=0.0)
            actual_melt = torch.min(snow_pack, melt)
            
            snow_pack = snow_pack + snowfall - actual_melt
            liquid_delivery[:, i] = rainfall + actual_melt
            
        return liquid_delivery

class PhysicsHybridEALSTM(nn.Module):
    def __init__(self, dyn_dim, stat_dim, num_basins, hidden_size, embed_dim=64):
        super().__init__()
        self.hidden_size = hidden_size
        self.basin_embed = nn.Embedding(num_basins, embed_dim)
        
        # Static Net predicts gates, states, and physical params
        self.static_net = nn.Sequential(
            nn.Linear(stat_dim + embed_dim, hidden_size),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, hidden_size * 2) 
        )
        
        # EA-LSTM components
        self.weight_dyn = nn.Linear(dyn_dim + 1, hidden_size * 3) # +1 for liquid delivery
        self.recurrent_dyn = nn.Linear(hidden_size, hidden_size * 3, bias=False)
        
        # Physical Param Head
        self.phys_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.GELU(),
            nn.Linear(32, 2),
            nn.Sigmoid()
        )
        
        self.snow_module = DifferentiableSnowPack()
        self.output_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Softplus()
        )
        
    def forward(self, x_d, x_s, b_idx, raw_p, raw_t):
        b, l, _ = x_d.shape
        s_feat = torch.cat([x_s, self.basin_embed(b_idx)], dim=-1)
        static_hidden = self.static_net(s_feat)
        i_gate_feat, phys_feat = torch.chunk(static_hidden, 2, dim=-1)
        
        # Learnable Physical Parameters (scaled to realistic ranges)
        # melt_factor: 0-8 mm/C/day, t_crit: -2 to 2 C
        p_params = self.phys_head(phys_feat)
        melt_factor = p_params[:, 0:1] * 8.0
        t_crit = (p_params[:, 1:2] - 0.5) * 4.0
        
        # Differentiable Physics Layer (using Daymet forcing index 0 and avg(2,3))
        liq = self.snow_module(raw_p, raw_t, melt_factor, t_crit).unsqueeze(-1)
        
        # EA-LSTM Cell
        i = torch.sigmoid(i_gate_feat)
        h = torch.zeros(b, self.hidden_size, device=x_d.device)
        c = torch.zeros(b, self.hidden_size, device=x_d.device)
        
        x_combined = torch.cat([x_d, liq], dim=-1)
        dyn_proj = self.weight_dyn(x_combined)
        
        h_out = []
        for t in range(l):
            gates = dyn_proj[:, t, :] + self.recurrent_dyn(h)
            f, g, o = torch.chunk(gates, 3, dim=-1)
            c = torch.sigmoid(f + 2.0) * c + i * torch.tanh(g)
            h = torch.sigmoid(o) * torch.tanh(c)
            h_out.append(h)
            
        h_seq = torch.stack(h_out, dim=1)
        return self.output_head(h_seq)

def prepare_features(dyn, aux):
    B, T, _ = dyn.shape
    # Add multi-scale rolling means for precipitation (Daymet, Maurer, NLDAS)
    p_idx = [0, 5, 10]
    feats = [dyn, aux]
    for w in [7, 30, 90]:
        p_roll = np.zeros((B, T, 3), dtype=np.float32)
        for i, idx in enumerate(p_idx):
            # Fast rolling mean
            p_series = dyn[:, :, idx]
            cs = np.cumsum(np.insert(p_series, 0, 0, axis=1), axis=1)
            p_roll[:, w-1:, i] = (cs[:, w:] - cs[:, :-w]) / w
        feats.append(p_roll)
    return np.concatenate(feats, axis=-1)

# --- Data Preparation ---
X_d_raw = prepare_features(DYNAMIC_INPUTS, AUX_FORCINGS)
X_s_raw = np.nan_to_num(STATIC_ATTRIBUTES, nan=0.0)

d_mean = X_d_raw[:, ERA_TRAIN_MASK, :].mean(axis=(0, 1), keepdims=True)
d_std = X_d_raw[:, ERA_TRAIN_MASK, :].std(axis=(0, 1), keepdims=True) + 1e-6
X_d_norm = torch.from_numpy((X_d_raw - d_mean) / d_std).to(DEVICE)

s_mean, s_std = X_s_raw.mean(axis=0), X_s_raw.std(axis=0) + 1e-6
X_s_norm = torch.from_numpy((X_s_raw - s_mean) / s_std).to(DEVICE)

# Raw inputs for the physics layer
RAW_P = torch.from_numpy(DYNAMIC_INPUTS[:, :, 0]).to(DEVICE) # Daymet P
RAW_T = torch.from_numpy((DYNAMIC_INPUTS[:, :, 2] + DYNAMIC_INPUTS[:, :, 3])/2).to(DEVICE) # Daymet T_avg

Y_gpu = torch.from_numpy(QOBS_TRAIN).to(DEVICE)
y_std_basin = np.array([np.nanstd(QOBS_TRAIN[b, ERA_TRAIN_MASK]) + 0.05 for b in range(len(BASINS))], dtype=np.float32)
Y_std_gpu = torch.from_numpy(y_std_basin).to(DEVICE)

train_indices = np.where(ERA_TRAIN_MASK)[0]
valid_starts = train_indices[train_indices >= (SEQ_LEN - 1)]
Q_PRED = np.zeros((len(BASINS), len(DATES)), dtype=np.float32)

# --- Training ---
for m_idx in range(NUM_MODELS):
    print(f"Training Model {m_idx+1}/{NUM_MODELS}...")
    torch.manual_seed(100 + m_idx)
    model = PhysicsHybridEALSTM(X_d_norm.shape[-1], X_s_norm.shape[-1], len(BASINS), HIDDEN_SIZE, EMBED_DIM).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        b_idx_arr = np.repeat(np.arange(len(BASINS)), SAMPLES_PER_BASIN)
        t_idx_arr = np.concatenate([np.random.choice(valid_starts, SAMPLES_PER_BASIN) for _ in range(len(BASINS))])
        perm = np.random.permutation(len(b_idx_arr))
        
        total_loss = 0
        for i in range(0, len(perm), BATCH_SIZE):
            idx = perm[i:i+BATCH_SIZE]
            batch_b, batch_t = torch.from_numpy(b_idx_arr[idx]).to(DEVICE), torch.from_numpy(t_idx_arr[idx]).to(DEVICE)
            
            # Temporal Window
            time_cols = batch_t.unsqueeze(1) + (torch.arange(SEQ_LEN, device=DEVICE) - (SEQ_LEN - 1))
            bx_d = X_d_norm[batch_b.unsqueeze(1), time_cols]
            bx_s = X_s_norm[batch_b]
            by = Y_gpu[batch_b.unsqueeze(1), time_cols]
            brp = RAW_P[batch_b.unsqueeze(1), time_cols]
            brt = RAW_T[batch_b.unsqueeze(1), time_cols]
            bstd = Y_std_gpu[batch_b].unsqueeze(1)
            
            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                preds = model(bx_d, bx_s, batch_b, brp, brt).squeeze(-1)
                # Basin-Normalized MSE Loss
                mask = ~torch.isnan(by)
                mask[:, :WARMUP_LEN] = False
                diff = (preds[mask] - by[mask]) / bstd.expand_as(preds)[mask]
                loss = F.mse_loss(diff, torch.zeros_like(diff))
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step()
        if epoch % 10 == 0:
            print(f"  Epoch {epoch:02d} | Loss: {total_loss/(len(perm)/BATCH_SIZE):.5f}")

    model.eval()
    with torch.no_grad():
        all_b = torch.arange(len(BASINS), device=DEVICE)
        for b_range in range(0, len(BASINS), 32):
            end = min(b_range + 32, len(BASINS))
            p_out = model(X_d_norm[b_range:end], X_s_norm[b_range:end], all_b[b_range:end], RAW_P[b_range:end], RAW_T[b_range:end])
            Q_PRED[b_range:end] += p_out.squeeze(-1).cpu().numpy() / NUM_MODELS

print("Inference Complete.")

Training Model 1/3...
  Epoch 10 | Loss: 0.14277
  Epoch 20 | Loss: 0.09450
  Epoch 30 | Loss: 0.07202
  Epoch 40 | Loss: 0.05927
  Epoch 50 | Loss: 0.05521
Training Model 2/3...
  Epoch 10 | Loss: 0.14828
  Epoch 20 | Loss: 0.09707
  Epoch 30 | Loss: 0.07117
  Epoch 40 | Loss: 0.05868
  Epoch 50 | Loss: 0.05584
Training Model 3/3...
  Epoch 10 | Loss: 0.15205
  Epoch 20 | Loss: 0.09954
  Epoch 30 | Loss: 0.07231
  Epoch 40 | Loss: 0.06459
  Epoch 50 | Loss: 0.05939
Inference Complete.


In [6]:
# ==============================================================================
# Immutable ERA Hill-Climb Evaluation (Strictly 1999-10-01 to 2003-09-30)
# ==============================================================================
_eval_npz = np.load(_find_file("era_eval_targets.npz"), allow_pickle=True)
_eval_raw_dates = np.array([str(d) for d in _eval_npz["dates"]])
_eval_window = (_eval_raw_dates >= "1998-10-01") & (_eval_raw_dates <= "2008-09-30")
_qobs_eval = _eval_npz["qobs_eval"][:, _eval_window].astype(np.float32)

assert "Q_PRED" in globals(), "Mutable cell must define Q_PRED!"
Q_PRED = np.asarray(Q_PRED, dtype=np.float32)
assert Q_PRED.shape == (len(BASINS), len(DATES)), (
    f"Expected Q_PRED shape {(len(BASINS), len(DATES))}, got {Q_PRED.shape}"
)
assert not np.any(np.isnan(Q_PRED[:, ERA_EVAL_MASK])) and not np.any(np.isinf(Q_PRED[:, ERA_EVAL_MASK])), (
    "Q_PRED contains NaN or Inf on ERA_EVAL_MASK (1999-10-01 to 2003-09-30)!"
)

def _compute_basin_nses(q_pred: np.ndarray, q_true: np.ndarray, mask: np.ndarray) -> np.ndarray:
    nses = []
    for i in range(len(BASINS)):
        yt = q_true[i, mask]
        yp = q_pred[i, mask]
        valid = ~np.isnan(yt)
        if np.sum(valid) < 30:
            continue
        yt_v = yt[valid]
        yp_v = yp[valid]
        denom = np.sum((yt_v - np.mean(yt_v)) ** 2)
        if denom > 1e-12:
            nses.append(1.0 - float(np.sum((yp_v - yt_v) ** 2) / denom))
    return np.asarray(nses, dtype=np.float64)

_nses_eval = _compute_basin_nses(Q_PRED, _qobs_eval, ERA_EVAL_MASK)
_median_nse = float(np.median(_nses_eval))
_mean_nse = float(np.mean(_nses_eval))
_q25, _q75 = float(np.percentile(_nses_eval, 25)), float(np.percentile(_nses_eval, 75))
_neg_count = int(np.sum(_nses_eval < 0.0))

print(f"=== ERA Evaluation Summary ({SPATIAL_REGIME} | {len(_nses_eval)} Basins | 1999-10-01 to 2003-09-30) ===")
print(f"Median NSE (1999-2003): {_median_nse:.5f} | Mean NSE: {_mean_nse:.5f} | IQR: [{_q25:.5f}, {_q75:.5f}] | Basins < 0: {_neg_count}")
print(f"Candidate metric to be maximized: {_median_nse:.6f}")

=== ERA Evaluation Summary (gauged | 531 Basins | 1999-10-01 to 2003-09-30) ===
Median NSE (1999-2003): 0.82043 | Mean NSE: 0.71903 | IQR: [0.71951, 0.86991] | Basins < 0: 16
Candidate metric to be maximized: 0.820428


In [7]:
# End Stage